In [2]:
import numpy as np
import pandas as pd

In [3]:
df_train = pd.read_csv('mnist_train.csv')

In [4]:
df_test = pd.read_csv('mnist_test.csv')

In [5]:
X_test = df_test.drop(columns=['label'])
y_test = df_test['label']

In [6]:
df_train

,label,1x1,1x2,1x3,1x4,1x5,1x6,1x7,1x8,1x9,...,28x19,28x20,28x21,28x22,28x23,28x24,28x25,28x26,28x27,28x28
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59996,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59997,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59998,6,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
X_train = df_train.drop(columns=['label'])
y_train = df_train['label']

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
# from lightgbm import LGBMClassifier

In [10]:
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# Optimized for SPEED - Reduced complexity
estimators_advanced = [
    ('rf', RandomForestClassifier(
        n_estimators=50,          # Reduced from 100
        max_depth=8,              # Reduced from 12
        min_samples_split=20,     # Increased for faster splits
        min_samples_leaf=10,      # Increased for faster splits
        max_features='sqrt',
        max_samples=0.7,
        random_state=42,
        n_jobs=-1
    )),
    ('mlp', MLPClassifier(
        hidden_layer_sizes=(64, 32),  # Reduced from (128, 64, 32)
        activation='relu',
        solver='adam',
        alpha=0.01,
        batch_size=512,           # Increased from 256 (faster)
        learning_rate_init=0.001,
        max_iter=80,              # Reduced from 200
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,      # Reduced from 20
        random_state=42,
        verbose=False
    )),
    ('knn', KNeighborsClassifier(
        n_neighbors=5,
        weights='distance',
        algorithm='auto',         # Faster than ball_tree for this size
        leaf_size=40,
        n_jobs=-1                 # Added for parallel processing
    ))
]

In [11]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

# Reduce the memory footprint
clf = StackingClassifier(
    estimators = estimators_advanced,
    n_jobs=1,  # IMPORTANT: Disable parallel processing (use 1 core)
    verbose=1,
    stack_method='predict_proba',
    passthrough=False
)

In [12]:
clf.fit(X_train, y_train)

/home/ayush-raiyani/anaconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (80) reached and the optimization hasn't converged yet.
  warnings.warn(
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    9.5s finished
/home/ayush-raiyani/anaconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (80) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/ayush-raiyani/anaconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (80) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/ayush-raiyani/anaconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (80) reached an

,estimators,"[('rf', ...), ('mlp', ...), ...]"
,final_estimator,None
,cv,None
,stack_method,'predict_proba'
,n_jobs,1
,passthrough,False
,verbose,1
,n_estimators,50
,criterion,'gini'
,max_depth,8
,min_samples_split,20


In [13]:
y_pred = clf.predict(X_test)

In [14]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.976

In [16]:
def quick_overfitting_check(clf, X_train, X_test, y_train, y_test):

    train_pred = clf.predict(X_train)
    test_pred = clf.predict(X_test)
    
    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)
    
    print(f"Training Accuracy: {train_acc:.4f}")
    print(f"Test Accuracy:     {test_acc:.4f}")
    print(f"Gap:               {train_acc - test_acc:.4f}")
    
    return train_acc, test_acc

# Quick check
train_acc, test_acc = quick_overfitting_check(clf, X_train, X_test, y_train, y_test)

Training Accuracy: 0.9998
Test Accuracy:     0.9760
Gap:               0.0238
